<a href="https://colab.research.google.com/github/imdann06/diabetes-risk-prediction/blob/mfernandaecheverri-tech-patch-1/division_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Partición Estratificada y Prevención de Data Leakage:

Se realiza una división estratificada del dataset (80% entrenamiento, 20% prueba) utilizando la variable objetivo Diabetes_Risk. La estratificación asegura que la    proporción de las clases (Low, Moderate, High) se mantenga intacta en ambos subconjuntos. Para garantizar la total ausencia de fuga de información (Data Leakage), todas las transformaciones de preprocesamiento se encadenan mediante un ColumnTransformer que ajusta sus parámetros (fit_transform) únicamente con X_train, aplicando posteriormente dicha transformación (transform) sobre X_test sin recalcular estadísticas.

2. Criterios para las Transformaciones Elegidas:

Imputación por Mediana (Variables Numéricas): Se opta por la mediana en lugar de la media para imputar los valores faltantes en las variables cuantitativas, debido a que la mediana es una medida de tendencia central robusta ante la presencia de datos atípicos (outliers) o distribuciones con sesgo.

Imputación por Moda (Variables Categóricas): Para las variables cualitativas con datos faltantes, se imputa con la moda (most frequent), preservando la categoría categórica de mayor prevalencia sin alterar la estructura cualitativa.

One-Hot Encoding (Variables Categóricas): Se transforma la información categórica no ordinal a una representación vectorial binaria (dummy). Se utiliza el parámetro handle_unknown='ignore' para prevenir errores en tiempo de inferencia ante categorías no observadas durante el entrenamiento en X_train.

Estandarización / StandardScaler (Variables Numéricas): Se escalan los atributos cuantitativos para que tengan una media de 0 y una desviación estándar de 1. Esto equipara la magnitud de todas las variables, previniendo que atributos con rangos numéricos elevados dominen de manera indebida sobre modelos sensibles a la escala (como regresión logística, máquinas de soporte vectorial o redes neuronales).

In [2]:
import zipfile
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

nombre_zip = 'diabetes_risk_prediction_dataset.zip.zip'

with zipfile.ZipFile(nombre_zip, 'r') as zip_ref:
    zip_ref.extractall('datos_descomprimidos') # Crea una carpeta con los datos
    print(" Archivo descompressión completada.")

# PASO 1: Cargar el CSV desde la carpeta extraída
df = pd.read_csv('datos_descomprimidos/diabetes_risk_prediction_dataset.csv')

# PASO 2: (División Train/Test sin Data Leakage)
X = df.drop(columns=['Diabetes_Risk'])
y = df['Diabetes_Risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n Dataset cargado y dividido exitosamente:")
print(f" Filas para entrenamiento (X_train): {X_train.shape[0]}")
print(f" Filas para prueba (X_test): {X_test.shape[0]}")

 Archivo descompressión completada.

 Dataset cargado y dividido exitosamente:
 Filas para entrenamiento (X_train): 40000
 Filas para prueba (X_test): 10000


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 1. VERIFICACIÓN Y SIMULACIÓN DE VALORES FALTANTES
nulos_existentes = X_train.isnull().sum().sum()

if nulos_existentes == 0:
    np.random.seed(42)
    n_samples = int(len(X_train) * 0.01)
    col_num = X_train.select_dtypes(include=[np.number]).columns[0]

    # Uso de .iloc para evitar la advertencia de pandas sobre copias
    idx_nan = np.random.choice(len(X_train), size=n_samples, replace=False)
    X_train.iloc[idx_nan, X_train.columns.get_loc(col_num)] = np.nan
    print(f"No se detectaron nulos. Se introdujo 1% de nulos controlados en '{col_num}'.")
else:
    print(f"El conjunto de entrenamiento ya contiene {nulos_existentes} valores faltantes reales.")
# 2. DEFINICIÓN DEL PIPELINE DE PREPROCESAMIENTO
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns

num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# 3. TRANSFORMACIÓN SIN DATA LEAKAGE
# fit_transform SOLO en Train / transform SOLO en Test
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print("\n Preprocesamiento completado con éxito:")
print(f" Forma final de X_train procesado: {X_train_prep.shape}")
print(f" Forma final de X_test procesado: {X_test_prep.shape}")

El conjunto de entrenamiento ya contiene 16766 valores faltantes reales.

 Preprocesamiento completado con éxito:
 Forma final de X_train procesado: (40000, 105)
 Forma final de X_test procesado: (10000, 105)
